# Hi-EF Phase 2: canonical residual preflight

This notebook performs contract tests and tiny smoke runs for `context`, `affect`, `interaction`, and `both`. It never evaluates the test partition. The smoke weights activate every implemented loss path but are **not research hyperparameters**, and smoke metrics must not be interpreted as experimental results.

Attach `ptrnghieu/hi-ef-features-v2`, enable a T4 GPU and Internet, then run all cells.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
FEATURES = Path('/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2')
OUTPUT = Path('/kaggle/working/canonical_residual_preflight')

if not (REPO / '.git').exists():
    subprocess.run([
        'git', 'clone', '--branch', 'experiments', '--single-branch',
        'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'experiments'], check=True)

assert (FEATURES / '01_00059.pt').is_file(), 'Feature dataset is not attached'
MANIFEST = REPO / 'experiments/manifests/source_folder_split_seed42.csv'
assert MANIFEST.is_file()
commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Ready at commit:', commit)

In [ ]:
command = [
    'python', str(REPO / 'experiments/run_canonical_preflight.py'),
    '--manifest', str(MANIFEST),
    '--features-dir', str(FEATURES),
    '--output-dir', str(OUTPUT),
]
subprocess.run(command, check=True)

In [ ]:
import json

summary_path = OUTPUT / 'canonical_preflight_summary.json'
summary = json.loads(summary_path.read_text())
assert summary['smoke_only'] is True
assert summary['research_interpretation_permitted'] is False
assert summary['test_evaluated'] is False
assert all(item['invariant_passed'] for item in summary['variants'].values())
print(json.dumps(summary, indent=2))
print('Download this file:', summary_path)